In [6]:
# 1) Carregamento dos dados
import pandas as pd

dados = pd.read_csv(
    'dados_IFData_2024_limpo.csv',
    sep=';',
    encoding='utf-8',
    decimal=',',
    thousands='.'
)

dados.head()

,Instituição,Código da instituição,Nome do conglomerado prudencial,Código do conglomerado financeiro,Código do conglomerado prudencial,TCB,TC,TI,Cidade,UF,Data,Ativo Total,Carteira de Crédito Classificada,Passivo Circulante e Exigível a Longo Prazo e Resultados de Exercícios Futuros,Captações,Patrimônio Líquido,Lucro Líquido,Número de Agências,Número de Postos de Atendimento
0,BANCO DO BRASIL S.A.,0,BB - PRUDENCIAL,49906.0,80329.0,b1,1,1,BRASILIA,DF,12/2024,2.304112e+09,1.035355e+09,2.123233e+09,1.823966e+09,180878517.0,17590402.0,3998,504
1,ITAÚ UNIBANCO S.A.,60701190,ITAU - PRUDENCIAL,10069.0,80099.0,b1,2,8,SAO PAULO,SP,12/2024,2.090515e+09,4.653437e+08,1.923957e+09,1.665638e+09,166558229.0,13515544.0,2012,1111
2,CAIXA ECONOMICA FEDERAL,360305,CAIXA ECONÔMICA FEDERAL - PRUDENCIAL,51626.0,80738.0,b1,1,6,BRASILIA,DF,12/2024,2.026392e+09,1.236308e+09,1.922424e+09,1.733095e+09,103968122.0,7311709.0,3258,917
3,BANCO BRADESCO S.A.,60746948,BRADESCO - PRUDENCIAL,10045.0,80075.0,b1,2,8,OSASCO,SP,12/2024,1.679912e+09,5.748721e+08,1.519425e+09,1.310203e+09,160486709.0,10158657.0,2298,918
4,BANCO SANTANDER (BRASIL) S.A.,90400888,SANTANDER - PRUDENCIAL,30379.0,80185.0,b1,3,8,SAO PAULO,SP,12/2024,1.264471e+09,3.647265e+08,1.174366e+09,8.907010e+08,90105019.0,6953930.0,2434,1496


In [9]:
# 2) Diagnóstico inicial do dataset

print(f"Linhas: {dados.shape[0]}")
print(f"Colunas: {dados.shape[1]}")

print("\n1. Tipos de dados:\n")
print(dados.dtypes)

print("\n2. Valores ausentes por coluna (top 10):\n")
print(dados.isna().sum().sort_values(ascending=False).head(10))

print("\n3. Resumo estatístico numérico:\n")
print(dados.describe(include='number').round(2))

print("\n4. Top 10 UFs por quantidade de instituições:\n")
print(dados['UF'].value_counts().head(10))

print("\n5. Top 10 instituições por Ativo Total:\n")
print(dados.nlargest(10, 'Ativo Total')[['Instituição', 'UF', 'Ativo Total']].to_string(index=False))

Linhas: 1585
Colunas: 19

1. Tipos de dados:

Instituição                                                                           str
Código da instituição                                                               int64
Nome do conglomerado prudencial                                                       str
Código do conglomerado financeiro                                                 float64
Código do conglomerado prudencial                                                 float64
TCB                                                                                   str
TC                                                                                  int64
TI                                                                                  int64
Cidade                                                                                str
UF                                                                                    str
Data                                                  

In [ ]:
# 3) Agregação por UF e rankings financeiros
uf_metrics = dados.groupby('UF').agg(
    total_instituicoes=('Instituição', 'count'),
    ativo_total=('Ativo Total', 'sum'),
    patrimonio_liquido=('Patrimônio Líquido', 'sum'),
    lucro_liquido=('Lucro Líquido', 'sum'),
    captacoes=('Captações', 'sum')
).sort_values('ativo_total', ascending=False)

print("\nIndicadores por UF (top 10):\n")
print(uf_metrics.head(10).round(2).to_string())

print("\nTop 10 instituições por Patrimônio Líquido:\n")
print(dados.nlargest(10, 'Patrimônio Líquido')[['Instituição', 'UF', 'Patrimônio Líquido']].to_string(index=False))

print("\nTop 10 instituições por Lucro Líquido:\n")
print(dados.nlargest(10, 'Lucro Líquido')[['Instituição', 'UF', 'Lucro Líquido']].to_string(index=False))

In [ ]:
# 4) Visualizações
import matplotlib.pyplot as plt

uf_top10 = uf_metrics.head(10).reset_index()
plt.figure(figsize=(12, 6))
plt.barh(uf_top10['UF'], uf_top10['ativo_total'] / 1_000_000, color='steelblue')
plt.title('Ativo Total por UF (em milhões)')
plt.xlabel('Ativo Total (R$ milhões)')
plt.ylabel('UF')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

pat_top10 = dados.nlargest(10, 'Patrimônio Líquido')[['Instituição', 'Patrimônio Líquido']].copy()
pat_top10['Patrimônio Líquido'] = pat_top10['Patrimônio Líquido'] / 1_000_000
plt.figure(figsize=(12, 6))
plt.barh(pat_top10['Instituição'], pat_top10['Patrimônio Líquido'], color='darkgreen')
plt.title('Top 10 instituições por Patrimônio Líquido (em milhões)')
plt.xlabel('Patrimônio Líquido (R$ milhões)')
plt.ylabel('Instituição')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

comparacao = dados.nlargest(10, 'Ativo Total')[['Instituição', 'Ativo Total', 'Patrimônio Líquido']].copy()
comparacao['Ativo Total'] = comparacao['Ativo Total'] / 1_000_000
comparacao['Patrimônio Líquido'] = comparacao['Patrimônio Líquido'] / 1_000_000
plt.figure(figsize=(12, 6))
plt.plot(comparacao['Instituição'], comparacao['Ativo Total'], marker='o', label='Ativo Total')
plt.plot(comparacao['Instituição'], comparacao['Patrimônio Líquido'], marker='s', label='Patrimônio Líquido')
plt.title('Comparação entre Ativo Total e Patrimônio Líquido (Top 10)')
plt.xticks(rotation=45, ha='right')
plt.ylabel('R$ milhões')
plt.legend()
plt.tight_layout()
plt.show()

# Conclusão da análise

Os resultados mostram que o sistema financeiro brasileiro analisado é fortemente concentrado no estado de São Paulo, que lidera em número de instituições, ativo total e captações. O DF também se destaca pela presença de grandes bancos e por um forte volume de ativos, embora em menor escala que SP.

Em termos de instituições, os maiores destaques são Banco do Brasil, Itaú Unibanco, Caixa, Bradesco e Santander, que dominam o ranking de ativo total e patrimônio líquido. O Itaú Unibanco Holding aparece como a instituição com maior patrimônio líquido, enquanto o Banco do Brasil e o Itaú Unibanco também se destacam em lucro líquido.

A análise também indica que alguns campos possuem valores ausentes, especialmente dados relacionados ao conglomerado prudencial e financeiro, o que deve ser tratado com atenção em uma etapa posterior de limpeza e padronização.

Em resumo, a base confirma a concentração regional e institucional do setor financeiro, com grande relevância dos grandes bancos e da região Sudeste, especialmente São Paulo.